# Layer Merging Comparison — GPT-2 / Kaggle

This notebook compares four methods for compressing two adjacent blocks, $L_i$ and $L_{i+1}$, into the equivalent of a single block under the same GPT-2 / WikiText-2 setting.

| Method | How two layers are compressed into one equivalent layer | Training |
|---|---|---|
| LinearMultiply | Approximate the two stages with affine maps and compose them as $A_1A_2$ | None (ridge regression only) |
| Remove | Remove $L_{i+1}$ and keep $L_i$ | None |
| Average | Replace $L_i$ and $L_{i+1}$ with one block obtained by a 50/50 average of corresponding parameters | None |
| NonlinearMultiply | Initialize one GPT2Block from $L_i$ and train it to approximate $L_{i+1}(L_i(h))$ | Hidden MSE + logit distillation, **15 epochs** |

For each method, all 11 adjacent pairs are compared on the validation set, and only the best pair for each method is locked as the final TEST candidate.

**Enable a GPU Accelerator in Kaggle and run the cells from top to bottom.** In `full` mode, NonlinearMultiply trains 11 pairs × 15 epochs, so execution takes considerably longer.


## 1. Kaggle Environment

This notebook is designed for **Kaggle Notebook** and uses Transformers 4.57.6, the same version as the original notebook.

Before running, check the following Kaggle Notebook settings:

- **Accelerator: GPU** (T4 / P100, etc.)
- **Internet: On** (required to download GPT-2, WikiText-2, and dependencies from Hugging Face)
- Run the cells from top to bottom.

Code, validation checks, training results, and ZIP files are saved under `/kaggle/working`.
If you have already imported a different version of `transformers` before running the dependency-installation cell, restart the Kaggle session and rerun the notebook from the beginning.


In [1]:
%pip install -q --disable-pip-version-check transformers==4.57.6 datasets==4.4.1 scipy==1.16.3 pandas==2.2.3 threadpoolctl==3.6.0


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 39.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 511.6/511.6 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 119.7/119.7 kB 7.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 201.0/201.0 kB 9.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 13.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 151.9/151.9 kB 6.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 38.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gradio 6.27.0 requires huggingface-hub<2.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
gcsfs 2025.12.0 requires fsspec==2025.12.0, but you have fsspec 2025.10.0 wh

## 2. Set Up the Implementation

The following cells only create Python files under `/kaggle/working`; the main experiment has not started yet.


In [2]:
%%writefile /kaggle/working/affine_core.py
"""Unchanged affine/evaluation utilities extracted from the supplied notebook.
Only functions needed by Nonlinear Composition Merging are included.
"""
from __future__ import annotations
import os
os.environ.setdefault("CUBLAS_WORKSPACE_CONFIG", ":4096:8")
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")
os.environ.setdefault("USE_TF", "0")
os.environ.setdefault("USE_FLAX", "0")
import math
import random
import time
import numpy as np
import scipy
from scipy.linalg import cho_factor, cho_solve
import torch
from torch import nn
import transformers
from transformers import AutoModelForCausalLM, AutoTokenizer

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.use_deterministic_algorithms(True)
    torch.backends.cudnn.benchmark = False
    torch.backends.cudnn.deterministic = True
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False


def _array(x):
    if torch.is_tensor(x):
        x = x.detach().cpu().numpy()
    a = np.asarray(x, dtype=np.float64)
    if a.ndim != 2 or not np.isfinite(a).all():
        raise ValueError("Expected a finite [tokens, features] matrix")
    return a


def _relative_reg(gram, strength):
    if strength <= 0:
        raise ValueError("A positive ridge is required")
    scale = float(np.trace(gram) / gram.shape[0])
    return strength * max(scale, np.finfo(np.float64).eps)


def fit_affine(X, Y, ridge=1e-4):
    """min ||XA+b-Y||_F^2 + lambda ||A||_F^2; bias unpenalized.

    lambda = ridge * trace(Xc.T Xc)/d, matching the previous experiment.
    No explicit inverse; CPU float64 is intentional for stable CCA/ridge fits.
    """
    X, Y = _array(X), _array(Y)
    if len(X) != len(Y) or len(X) < 2:
        raise ValueError("X and Y need the same N >= 2")
    mx, my = X.mean(0), Y.mean(0)
    xc, yc = X - mx, Y - my
    gram = xc.T @ xc
    reg = _relative_reg(gram, ridge)
    A = cho_solve(cho_factor(gram + reg * np.eye(X.shape[1])), xc.T @ yc)
    b = my - mx @ A
    return A, b


class AffineMergedBlock(nn.Module):
    """GPT2Block ABI for transformers==4.57.6, no KV cache.

    This already approximates the WHOLE two-block map; do not add x again.
    """
    def __init__(self, A, b, device=None, dtype=torch.float32):
        super().__init__()
        d = A.shape[0]
        self.linear = nn.Linear(d, d, device=device, dtype=dtype)
        with torch.no_grad():
            self.linear.weight.copy_(torch.as_tensor(np.array(A.T, copy=True)))
            self.linear.bias.copy_(torch.as_tensor(np.array(b, copy=True)))

    def forward(self, hidden_states, past_key_values=None, cache_position=None,
                attention_mask=None, head_mask=None, encoder_hidden_states=None,
                encoder_attention_mask=None, use_cache=False, output_attentions=False,
                **kwargs):
        if use_cache or past_key_values is not None:
            raise ValueError("Affine merge supports use_cache=False and no past_key_values")
        if encoder_hidden_states is not None:
            raise ValueError("Cross-attention is outside this GPT-2 experiment")
        output = (self.linear(hidden_states),)
        return output + (None,) if output_attentions else output


def parameter_count(model):
    return sum(p.numel() for p in model.parameters())  # tied weights counted once


def make_state_windows(ids, total_tokens, chunk_size, seed=None):
    """Disjoint source windows; random order for TRAIN, sequential for VAL."""
    ids = ids.detach().cpu().flatten()
    if len(ids) < total_tokens or total_tokens < 2 or chunk_size < 2:
        raise ValueError("Invalid number of state tokens / chunk size")
    starts = np.arange(0, len(ids), chunk_size)
    if seed is not None:
        np.random.default_rng(seed).shuffle(starts)
    windows, remaining = [], total_tokens
    for start in starts:
        end = min(int(start) + chunk_size, len(ids), int(start) + remaining)
        if end <= start:
            continue
        windows.append(ids[int(start):end].clone())
        remaining -= end - int(start)
        if remaining == 0:
            break
    if remaining:
        raise ValueError("Insufficient tokens")
    return windows


@torch.inference_mode()
def collect_pair_states(model, windows, L):
    """Hooks capture pre-ln_f block outputs, including pair (10,11)."""
    if model.training:
        raise ValueError("Collect states with model.eval()")
    d = model.config.n_embd
    n = sum(w.numel() for w in windows)
    states = [np.empty((n, d), np.float32) for _ in range(3)]
    position = [0, 0, 0]
    def store(index, tensor):
        a = tensor.detach().reshape(-1, d).float().cpu().numpy()
        start = position[index]
        states[index][start:start + len(a)] = a
        position[index] += len(a)
    handles = [
        model.transformer.h[L].register_forward_pre_hook(lambda m, inp: store(0, inp[0])),
        model.transformer.h[L].register_forward_hook(lambda m, inp, out: store(1, out[0])),
        model.transformer.h[L + 1].register_forward_hook(lambda m, inp, out: store(2, out[0])),
    ]
    try:
        device = next(model.parameters()).device
        for window in windows:
            # No vocabulary logits needed while collecting states.
            model.transformer(window[None].to(device), use_cache=False, return_dict=True)
    finally:
        for handle in handles:
            handle.remove()
    if position != [n, n, n]:
        raise RuntimeError(f"Incomplete activation collection: {position}")
    return tuple(states)


@torch.inference_mode()
def measure_latency(model, input_ids, repeats=10, warmup=5):
    """Batch=1 full LM forward, no KV cache, excludes CPU->GPU transfer."""
    if repeats < 1:
        raise ValueError("Latency repeats must be positive")
    device = next(model.parameters()).device
    x = input_ids.flatten()[None].to(device)
    def sync():
        if device.type == "cuda":
            torch.cuda.synchronize(device)
    model.eval()
    for _ in range(warmup):
        model(x, use_cache=False)
    sync()
    samples = []
    for _ in range(repeats):
        start = time.perf_counter()
        model(x, use_cache=False)
        sync()
        samples.append(1000 * (time.perf_counter() - start))
    return {"Latency_ms": float(np.median(samples)),
            "Latency_Mean_ms": float(np.mean(samples)),
            "Latency_P90_ms": float(np.quantile(samples, 0.9))}


def check_transformers_version():
    if transformers.__version__ != "4.57.6":
        raise RuntimeError("Install transformers==4.57.6 and restart the Kaggle session before importing.")



Writing /kaggle/working/affine_core.py


FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/working/affine_core.py'

In [ ]:
%%writefile /kaggle/working/evaluation_protocol.py
"""Exact sliding/legacy evaluator from merging_comparison_colab.ipynb."""
import json
import math
from pathlib import Path
import numpy as np
import torch

def target_mask(length, context, protocol):
    """Global positions whose next-token losses belong in the denominator."""
    if context < 2 or length < 2:
        raise ValueError("Need context and token count >= 2")
    mask = torch.zeros(length, dtype=torch.bool)
    if protocol == "sliding":
        mask[1:] = True
    elif protocol == "legacy":
        for start in range(0, (length//context)*context, context):
            mask[start+1:start+context] = True
    else:
        raise ValueError("protocol must be sliding or legacy")
    return mask


def protocol_windows(ids, context, stride, protocol="sliding", score_mask=None):
    ids = ids.detach().cpu().flatten()
    targets = target_mask(len(ids), context, protocol)
    if score_mask is not None:
        if score_mask.shape != targets.shape:
            raise ValueError("score_mask must match the token sequence")
        targets &= score_mask.cpu().bool()
    if not targets.any():
        raise ValueError("No scored tokens; legacy requires a full context window")
    if protocol == "sliding":
        if not 1 <= stride < context:
            raise ValueError("Sliding evaluation needs 1 <= stride < context")
        previous_end = 0
        for start in range(0, len(ids), stride):
            end = min(start+context, len(ids))
            x = ids[start:end][None]
            labels = x.clone()
            if previous_end > start:
                labels[:, :previous_end-start] = -100
            labels[:, ~targets[start:end]] = -100
            count = int((labels[:, 1:] != -100).sum())
            if count:
                yield x, labels, count
            previous_end = end
            if end == len(ids):
                break
    else:
        for start in range(0, (len(ids)//context)*context, context):
            x = ids[start:start+context][None]
            labels = x.clone()
            labels[:, ~targets[start:start+context]] = -100
            count = int((labels[:, 1:] != -100).sum())
            if count:
                yield x, labels, count


@torch.inference_mode()
def calculate_ppl(model, ids, context=512, stride=256, protocol="sliding", score_mask=None):
    """Exactly reproduce the old evaluator, or use the current sliding one."""
    model.eval()
    device = next(model.parameters()).device
    total_nll, total_count = 0.0, 0
    for x, labels, count in protocol_windows(ids, context, stride, protocol, score_mask):
        output = model(input_ids=x.to(device), labels=labels.to(device),
                       use_cache=False, return_dict=True)
        loss = float(output.loss)
        if not math.isfinite(loss):
            raise ArithmeticError("Non-finite PPL loss")
        total_nll += loss*count
        total_count += count
    expected = target_mask(ids.numel(), context, protocol)
    if score_mask is not None:
        expected &= score_mask.cpu().bool()
    if total_count != int(expected.sum()):
        raise AssertionError("Incorrect next-token denominator")
    nll = total_nll/total_count
    return {"PPL": math.exp(nll) if nll < 700 else float("inf"),
            "NLL": nll, "Predicted_Tokens": total_count}


def _write_json(path, value):
    def normalize(x):
        if isinstance(x, dict):
            return {k: normalize(v) for k, v in x.items()}
        if isinstance(x, (list, tuple)):
            return [normalize(v) for v in x]
        if isinstance(x, np.generic):
            return normalize(x.item())
        if isinstance(x, float) and not math.isfinite(x):
            return None
        return x
    Path(path).write_text(json.dumps(normalize(value), ensure_ascii=False, indent=2,
                                    allow_nan=False), encoding="utf-8")



In [ ]:
%%writefile /kaggle/working/nonlinear_composition.py
"""Four-way adjacent-layer merging comparison for GPT-2.

Methods:
- LinearMultiply: compose two fitted affine maps.
- Remove: keep L_i and remove L_{i+1}.
- Average: replace L_i,L_{i+1} with a 50/50 parameter-average GPT2Block.
- NonlinearMultiply: initialize one GPT2Block from L_i and train it to
  approximate L_{i+1}(L_i(h)) using hidden-state and logit distillation.

Only NonlinearMultiply is trained. No cached decoding.
"""
from __future__ import annotations
import copy
import gc
import hashlib
import json
import math
import platform
from contextlib import contextmanager
from dataclasses import asdict, dataclass, replace
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.checkpoint import checkpoint
from threadpoolctl import threadpool_limits
import affine_core as sm
import evaluation_protocol as ep

METHODS = ("LinearMultiply", "Remove", "Average", "NonlinearMultiply")


@dataclass(frozen=True)
class Config:
    model_id: str = "openai-community/gpt2"
    model_revision: str = "main"
    dataset_revision: str = "main"
    seed: int = 42
    mode: str = "quick"
    context_length: int = 512
    eval_stride: int = 256
    eval_tokens: int | None = 10000
    hidden_eval_tokens: int = 4096
    local_tokens: int = 4096
    kd_tokens: int = 4096
    holdout_tokens: int = 2048
    epochs: int = 10
    patience: int = 15
    grad_accumulation: int = 1
    lr_transformer: float = 3e-5
    weight_decay: float = 0.01
    max_grad_norm: float = 1.0
    lambda_hidden: float = 1.0
    lambda_kl: float = 1.0
    temperature: float = 2.0
    epsilon: float = 1e-8
    affine_ridge: float = 1e-4
    kl_chunk_tokens: int = 64
    cpu_threads: int = 2
    latency_seq_len: int = 256
    latency_repeats: int = 10
    output_dir: str = "/kaggle/working/layer_merging_4way_results"


def make_config(mode="quick", **kwargs):
    if mode not in ("quick", "full"):
        raise ValueError("mode must be quick or full")
    cfg = Config() if mode == "quick" else Config(mode="full", local_tokens=20480,
        kd_tokens=20480, holdout_tokens=4096, epochs=15, eval_tokens=None,
        hidden_eval_tokens=8192, latency_repeats=30)
    return replace(cfg, **kwargs)


def validate_config(cfg):
    if not 1 <= cfg.eval_stride < cfg.context_length:
        raise ValueError("Use overlapping sliding evaluation")
    for key in ("local_tokens", "kd_tokens", "holdout_tokens"):
        n = getattr(cfg, key)
        if n < cfg.context_length or n % cfg.context_length:
            raise ValueError(f"{key} must be a positive multiple of context_length")
    for key in ("epochs", "patience", "grad_accumulation", "kl_chunk_tokens",
                "latency_seq_len", "latency_repeats", "hidden_eval_tokens", "cpu_threads"):
        if getattr(cfg, key) < 1:
            raise ValueError(f"{key} must be positive")
    if cfg.temperature <= 0 or cfg.epsilon <= 0 or cfg.affine_ridge <= 0:
        raise ValueError("temperature, epsilon, affine_ridge must be positive")
    if cfg.lambda_hidden < 0 or cfg.lambda_kl < 0 or cfg.lambda_hidden + cfg.lambda_kl == 0:
        raise ValueError("Loss weights must be nonnegative and not both zero")
    if cfg.lr_transformer <= 0 or cfg.max_grad_norm <= 0 or cfg.weight_decay < 0:
        raise ValueError("Invalid optimizer configuration")


def average_blocks(block_a, block_b):
    """Return a deep-copied GPT2Block whose floating parameters are averaged 50/50."""
    merged = copy.deepcopy(block_a)
    a_state = block_a.state_dict()
    b_state = block_b.state_dict()
    if a_state.keys() != b_state.keys():
        raise ValueError("Adjacent GPT2Blocks have incompatible state_dict keys")
    averaged = {}
    for key in a_state:
        a, b = a_state[key], b_state[key]
        if a.shape != b.shape:
            raise ValueError(f"Cannot average mismatched tensor {key}: {a.shape} vs {b.shape}")
        if torch.is_floating_point(a) or torch.is_complex(a):
            averaged[key] = (a + b) * 0.5
        else:
            if not torch.equal(a, b):
                raise ValueError(f"Non-floating buffer differs between adjacent blocks: {key}")
            averaged[key] = a.clone()
    merged.load_state_dict(averaged)
    return merged


def build_student(teacher, pair, method, cfg, affine=None):
    if method not in METHODS or not 0 <= pair < len(teacher.transformer.h)-1:
        raise ValueError("Invalid method or pair")
    student = copy.deepcopy(teacher).eval().requires_grad_(False)
    blocks = list(student.transformer.h)
    ref = next(student.parameters())

    if method == "LinearMultiply":
        if affine is None:
            d = student.config.n_embd
            affine = (np.eye(d), np.zeros(d))  # overwritten when loading checkpoint
        merged = sm.AffineMergedBlock(*affine, device=ref.device, dtype=ref.dtype)
    elif method == "Remove":
        # Remove L_{i+1}; retain L_i as the one-block replacement.
        merged = blocks[pair]
    elif method == "Average":
        # Parameter-wise 50/50 average of L_i and L_{i+1}.
        merged = average_blocks(blocks[pair], blocks[pair+1]).to(ref.device, ref.dtype)
    else:  # NonlinearMultiply
        # Start from L_i, then learn one nonlinear Transformer block that approximates
        # the original two-block composition L_{i+1}(L_i(h)).
        merged = blocks[pair]

    student.transformer.h = nn.ModuleList(blocks[:pair] + [merged] + blocks[pair+2:])
    student.config.n_layer = len(student.transformer.h)
    student.config.use_cache = student.generation_config.use_cache = False

    trainable = method == "NonlinearMultiply"
    merged.requires_grad_(trainable)
    student.eval()  # disables dropout but not autograd
    expected = {id(p) for p in merged.parameters()} if trainable else set()
    assert {id(p) for p in student.parameters() if p.requires_grad} == expected
    assert len(student.transformer.h) == len(teacher.transformer.h)-1
    return student


def parameter_digest(model, excluded_prefix=None):
    digest = hashlib.sha256()
    for name, p in model.named_parameters():
        if excluded_prefix and name.startswith(excluded_prefix):
            continue
        digest.update(name.encode())
        digest.update(p.detach().cpu().contiguous().numpy().tobytes())
    return digest.hexdigest()


def split_train(ids, cfg):
    """Random disjoint source windows, exactly preserving old affine TRAIN.

    LOCAL uses the same seed/order/function as previous affine fitting.
    KD and HOLDOUT take subsequent full windows of that same permutation.
    Each window resets context; no prefix crosses a region boundary.
    """
    counts = [cfg.local_tokens, cfg.kd_tokens, cfg.holdout_tokens]
    # Preserve old make_state_windows order (including partial final window).
    starts = np.arange(0, len(ids), cfg.context_length)
    np.random.default_rng(cfg.seed).shuffle(starts)
    needed = sum(counts)//cfg.context_length
    if len(starts) < needed:
        raise ValueError("Not enough disjoint TRAIN windows")
    chosen = starts[:needed]
    if any(int(s)+cfg.context_length > len(ids) for s in chosen):
        raise ValueError("Selected a short final TRAIN window; choose another seed or use more data")
    parts, regions, offset = {}, [], 0
    for name, count in zip(("local", "kd", "holdout"), counts):
        subset = chosen[offset:offset+count//cfg.context_length]
        parts[name] = [ids[int(s):int(s)+cfg.context_length].clone() for s in subset]
        regions.extend({"Split": name, "Start": int(s), "End": int(s)+cfg.context_length} for s in subset)
        offset += len(subset)
    old = sm.make_state_windows(ids, cfg.local_tokens, cfg.context_length, cfg.seed)
    assert all(torch.equal(a, b) for a, b in zip(parts["local"], old)) and len(old) == len(parts["local"])
    assert len(set(int(s) for s in chosen)) == needed
    return parts, pd.DataFrame(regions)


@contextmanager
def capture_pair(model, pair):
    states = {}
    handles = [model.transformer.h[pair].register_forward_pre_hook(
        lambda m, args: states.update(h=args[0])),
        model.transformer.h[pair+1].register_forward_hook(
        lambda m, args, out: states.update(target=out[0]))]
    try:
        yield states
    finally:
        for handle in handles:
            handle.remove()


def hidden_loss(student_h, teacher_h, cfg):
    if student_h.shape != teacher_h.shape:
        raise AssertionError("Teacher/student hidden shape mismatch")
    raw = (student_h.float()-teacher_h.float()).square().mean()
    return raw/(teacher_h.float().square().mean()+cfg.epsilon), raw


def distillation_kl(student_logits, teacher_logits, cfg):
    """tau^2 * mean_tokens KL(teacher || student), vocabulary summed.

    Exclude last position: each scored logit predicts an observed next token.
    Full vocabulary retained. Checkpoint chunks to limit autograd memory.
    Teacher tensors originate in no_grad(), never inference_mode().
    """
    if student_logits.shape != teacher_logits.shape or student_logits.shape[1] < 2:
        raise ValueError("Expected matching [B,T,V] logits, T >= 2")
    s = student_logits[:, :-1].reshape(-1, student_logits.shape[-1])
    t = teacher_logits[:, :-1].reshape_as(s).detach()
    def chunk_loss(a, b):
        logp = F.log_softmax(b.float()/cfg.temperature, dim=-1)
        logq = F.log_softmax(a.float()/cfg.temperature, dim=-1)
        return (logp.exp()*(logp-logq)).sum()
    total = s.new_zeros((), dtype=torch.float32)
    for start in range(0, len(s), cfg.kl_chunk_tokens):
        a, b = s[start:start+cfg.kl_chunk_tokens], t[start:start+cfg.kl_chunk_tokens]
        part = checkpoint(chunk_loss, a, b, use_reentrant=False) if a.requires_grad else chunk_loss(a, b)
        total = total + part
    return cfg.temperature**2 * total/len(s)


def local_objective(teacher, student, pair, ids, cfg):
    with torch.no_grad(), capture_pair(teacher, pair) as states:
        teacher.transformer(ids, use_cache=False, return_dict=True)
    h, target = states["h"], states["target"]
    prediction = student.transformer.h[pair](h, use_cache=False)[0]
    if h.shape != target.shape:
        raise AssertionError("Pair input/output dimension changed")
    return hidden_loss(prediction, target, cfg)


def kd_objective(teacher, student, ids, cfg):
    with torch.no_grad():
        teacher_logits = teacher(ids, use_cache=False, return_dict=True).logits
    student_logits = student(ids, use_cache=False, return_dict=True).logits
    return distillation_kl(student_logits, teacher_logits, cfg)


@torch.no_grad()
def reconstruction_metrics(teacher, student, pair, windows, cfg):
    """Diagnostic windows only; raw MSE weighted by elements, KL by tokens."""
    device = next(teacher.parameters()).device
    sqerr = energy = kl_sum = 0.0
    elements = tokens = 0
    normalized = []
    for window in windows:
        ids = window[None].to(device)
        with capture_pair(teacher, pair) as states:
            t = teacher(ids, use_cache=False, return_dict=True).logits
        sh = student.transformer.h[pair](states["h"], use_cache=False)[0]
        hn, raw = hidden_loss(sh, states["target"], cfg)
        sqerr += float(raw)*sh.numel()
        energy += float(states["target"].float().square().sum())
        elements += sh.numel()
        normalized.append(float(hn))
        logits = student(ids, use_cache=False, return_dict=True).logits
        count = ids.shape[1]-1
        kl_sum += float(distillation_kl(logits, t, cfg))*count
        tokens += count
    raw = sqerr/elements
    kl = kl_sum/tokens
    # Train/holdout use mean of per-window normalized losses. Also report
    # globally normalized MSE for interpretation; these need not be identical.
    return {"Hidden_MSE": raw, "Normalized_Hidden_MSE": raw/(energy/elements+cfg.epsilon),
            "Mean_Window_Normalized_Hidden_MSE": float(np.mean(normalized)),
            "Teacher_KL": kl, "Teacher_KL_Unscaled": kl/cfg.temperature**2,
            "Diagnostic_Tokens": sum(len(w) for w in windows), "KL_Predicted_Tokens": tokens}


def cpu_state(block):
    return {key: value.detach().cpu().clone() for key, value in block.state_dict().items()}


def train_student(teacher, student, pair, method, parts, cfg, history_path=None):
    """Two disjoint minibatches per step; accumulate their weighted gradients.

    LOCAL contributes hidden loss; KD contributes logit loss. Both gradients
    update the same block at the same optimizer step. No validation access.
    Batch size is one full sequence; only NonlinearMultiply is optimized.
    """
    block = student.transformer.h[pair]
    frozen_before = parameter_digest(student, f"transformer.h.{pair}.")
    if method != "NonlinearMultiply":
        raise ValueError("Only NonlinearMultiply is trainable")
    optimizer = torch.optim.AdamW(block.parameters(), lr=cfg.lr_transformer,
                                 weight_decay=cfg.weight_decay)
    device = next(student.parameters()).device
    history, best_loss, best_epoch, best_state, stale = [], math.inf, 0, None, 0
    steps = max(len(parts["local"]), len(parts["kd"]))

    def holdout_row(epoch, train=None):
        metric = reconstruction_metrics(teacher, student, pair, parts["holdout"], cfg)
        h, k = metric["Mean_Window_Normalized_Hidden_MSE"], metric["Teacher_KL"]
        row = {"Pair": f"{pair}-{pair+1}", "Method": method, "Epoch": epoch,
               "Holdout_Hidden_Loss": h, "Holdout_KL": k,
               "Holdout_Total_Loss": cfg.lambda_hidden*h+cfg.lambda_kl*k}
        row.update(train or {"Train_Hidden_Loss": None, "Train_KL": None, "Train_Total_Loss": None})
        if not math.isfinite(row["Holdout_Total_Loss"]):
            raise ArithmeticError("Non-finite HOLDOUT loss")
        return row

    for epoch in range(cfg.epochs+1):
        train = None
        if epoch:
            rng = np.random.default_rng(cfg.seed + pair*1009 + epoch)
            order_local = rng.permutation(len(parts["local"]))
            order_kd = rng.permutation(len(parts["kd"]))
            hsum = ksum = 0.0
            optimizer.zero_grad(set_to_none=True)
            for step in range(steps):
                # Correct scaling even for a short final accumulation group.
                group_start = (step//cfg.grad_accumulation)*cfg.grad_accumulation
                divisor = min(cfg.grad_accumulation, steps-group_start)
                local = parts["local"][order_local[step % len(order_local)]][None].to(device)
                kd = parts["kd"][order_kd[step % len(order_kd)]][None].to(device)
                hn, raw = local_objective(teacher, student, pair, local, cfg)
                if not torch.isfinite(hn):
                    raise ArithmeticError("Non-finite hidden loss")
                hsum += float(hn.detach())
                if cfg.lambda_hidden:
                    (cfg.lambda_hidden*hn/divisor).backward()
                del hn, raw
                kl = kd_objective(teacher, student, kd, cfg)
                if not torch.isfinite(kl):
                    raise ArithmeticError("Non-finite KL")
                ksum += float(kl.detach())
                if cfg.lambda_kl:
                    (cfg.lambda_kl*kl/divisor).backward()
                del kl
                if (step+1) % cfg.grad_accumulation == 0 or step+1 == steps:
                    norm = torch.nn.utils.clip_grad_norm_(block.parameters(), cfg.max_grad_norm,
                                                          error_if_nonfinite=True)
                    optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
            h, k = hsum/steps, ksum/steps
            train = {"Train_Hidden_Loss": h, "Train_KL": k,
                     "Train_Total_Loss": cfg.lambda_hidden*h + cfg.lambda_kl*k}
        row = holdout_row(epoch, train)
        history.append(row)
        if history_path:
            pd.DataFrame(history).to_csv(history_path, index=False)
        score = row["Holdout_Total_Loss"]
        if score < best_loss:
            best_loss, best_epoch, best_state, stale = score, epoch, cpu_state(block), 0
        else:
            stale += 1
        print(f"  {method} epoch={epoch} holdout={score:.6f} best={best_epoch}", flush=True)
        if epoch and stale >= cfg.patience:
            break
    block.load_state_dict(best_state)
    assert parameter_digest(student, f"transformer.h.{pair}.") == frozen_before, "Frozen weights changed"
    return history, best_epoch


def fit_affine_multiply(teacher, windows, pair, cfg):
    # Same activation collection, ridge regression, and composition as input notebook.
    X, Y1, Y2 = sm.collect_pair_states(teacher, windows, pair)
    with threadpool_limits(limits=cfg.cpu_threads):
        A1, b1 = sm.fit_affine(X, Y1, cfg.affine_ridge)
        A2, b2 = sm.fit_affine(Y1, Y2, cfg.affine_ridge)
        return A1@A2, b1@A2+b2


def save_checkpoint(path, teacher, student, pair, method, cfg, epoch):
    torch.save({"format_version": 1, "method": method, "pair": pair,
                "model_id": cfg.model_id,
                "model_revision": getattr(teacher.config, "_commit_hash", None) or cfg.model_revision,
                "original_layers": len(teacher.transformer.h), "config": asdict(cfg),
                "best_epoch": epoch, "state_dict": cpu_state(student.transformer.h[pair])}, path)


def load_checkpoint(path, device=None, teacher=None):
    saved = torch.load(path, map_location="cpu", weights_only=True)
    cfg = Config(**saved["config"])
    if teacher is None:
        teacher = sm.AutoModelForCausalLM.from_pretrained(saved["model_id"],
            revision=saved["model_revision"], attn_implementation="eager", dtype=torch.float32
        ).to(device or ("cuda" if torch.cuda.is_available() else "cpu")).eval().requires_grad_(False)
    if len(teacher.transformer.h) != saved["original_layers"]:
        raise ValueError("Checkpoint base model layer count differs")
    student = build_student(teacher, saved["pair"], saved["method"], cfg)
    student.transformer.h[saved["pair"]].load_state_dict(saved["state_dict"])
    return student.eval().requires_grad_(False)


def cleanup():
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


def evaluate_ppl(model, ids, cfg):
    return ep.calculate_ppl(model, ids, cfg.context_length, cfg.eval_stride, protocol="sliding")


def run_experiment(cfg=None, run_test=False):
    cfg = cfg or make_config()
    validate_config(cfg)
    sm.check_transformers_version()
    sm.set_seed(cfg.seed)
    torch.set_num_threads(cfg.cpu_threads)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%S_%fZ")
    out = Path(cfg.output_dir)/f"merging4_{cfg.mode}_{stamp}"
    out.mkdir(parents=True, exist_ok=False)
    ep._write_json(out/"configuration.json", asdict(cfg))
    ep._write_json(out/"protocol.json", {"name": "Four-way Adjacent-Layer Merging", "run_test": run_test,
        "methods": METHODS, "evaluation": "float32/eager/sliding/no KV cache",
        "epoch_selection": f"NonlinearMultiply: best checkpoint from epochs 0..{cfg.epochs} by TRAIN_HOLDOUT normalized hidden MSE + temperature-scaled KL",
        "pair_selection": "VALIDATION NLL only; TEST never selects",
        "kl": "tau^2 KL(teacher || student), all vocabulary, positions 0..T-2",
        "dropout": "disabled for teacher and all candidates; only NonlinearMultiply is trained",
        "training": "NonlinearMultiply only: one LOCAL and one disjoint KD sequence per microstep",
        "limitations": "NonlinearMultiply uses distillation/training while the three baselines do not; "
                       "Average is a raw 50/50 parameter average and Remove retains L_i."})
    import datasets
    dataset = datasets.load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", revision=cfg.dataset_revision)
    tokenizer = sm.AutoTokenizer.from_pretrained(cfg.model_id, revision=cfg.model_revision)
    tokenizer.model_max_length = 10**12
    def tokenize(split, limit=None):
        text = "\n\n".join(t for t in dataset[split]["text"] if t and t.strip())
        ids = tokenizer(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
        return ids if limit is None else ids[:limit]
    train_ids = tokenize("train")
    val_ids = tokenize("validation", cfg.eval_tokens)
    parts, regions = split_train(train_ids, cfg)
    regions.to_csv(out/"train_regions.csv", index=False)
    torch.save({"parts": parts, "validation_ids": val_ids}, out/"data_tokens.pt")
    teacher = sm.AutoModelForCausalLM.from_pretrained(cfg.model_id, revision=cfg.model_revision,
        attn_implementation="eager", dtype=torch.float32).to(device).eval().requires_grad_(False)
    teacher.config.use_cache = teacher.generation_config.use_cache = False
    if max(cfg.context_length, cfg.latency_seq_len) > teacher.config.n_positions:
        raise ValueError("Sequence length exceeds GPT-2 context")
    layers = len(teacher.transformer.h)
    if cfg.model_id == "openai-community/gpt2" and layers != 12:
        raise AssertionError("Expected 12 GPT-2 blocks")
    teacher_digest = parameter_digest(teacher)
    ep._write_json(out/"environment.json", {"python": platform.python_version(),
        "torch": str(torch.__version__), "transformers": sm.transformers.__version__,
        "numpy": np.__version__, "pandas": pd.__version__, "scipy": sm.scipy.__version__,
        "datasets": datasets.__version__, "device": str(device),
        "gpu": torch.cuda.get_device_name(device) if device.type == "cuda" else None,
        "model_resolved_revision": getattr(teacher.config, "_commit_hash", None) or cfg.model_revision,
        "dataset_requested_revision": cfg.dataset_revision,
        "dataset_fingerprints": {name: part._fingerprint for name, part in dataset.items()},
        "teacher_parameter_sha256": teacher_digest})
    latency_ids = val_ids[:cfg.latency_seq_len]
    baseline = {**evaluate_ppl(teacher, val_ids, cfg), "Parameters": sm.parameter_count(teacher),
                **sm.measure_latency(teacher, latency_ids, cfg.latency_repeats)}
    ep._write_json(out/"baseline_validation.json", baseline)
    diag_windows = sm.make_state_windows(val_ids, min(cfg.hidden_eval_tokens, len(val_ids)), cfg.context_length)
    diag_windows = [w for w in diag_windows if len(w) >= 2]
    print(f"Device: {device}; output: {out}\nOriginal validation: {baseline}", flush=True)
    rows, histories, best = [], [], {}
    for pair in range(layers-1):
        print(f"\n[{pair+1}/{layers-1}] pair {pair}-{pair+1}", flush=True)
        affine = fit_affine_multiply(teacher, parts["local"], pair, cfg)
        for method in METHODS:
            sm.set_seed(cfg.seed + 1009*pair)  # stable across timing/order changes
            student = build_student(teacher, pair, method, cfg, affine)
            epoch = None
            if method == "NonlinearMultiply":
                hist, epoch = train_student(teacher, student, pair, method, parts, cfg,
                    out/f"history_{method}_{pair}_{pair+1}.csv")
                histories.extend(hist)
                pd.DataFrame(histories).to_csv(out/"training_history.csv", index=False)
            metric = evaluate_ppl(student, val_ids, cfg)
            reconstruction = reconstruction_metrics(teacher, student, pair, diag_windows, cfg)
            reference = sm.measure_latency(teacher, latency_ids, cfg.latency_repeats)
            latency = sm.measure_latency(student, latency_ids, cfg.latency_repeats)
            params = sm.parameter_count(student)
            row = {"Pair": f"{pair}-{pair+1}", "Pair_Start": pair, "Method": method,
                "Validation_PPL": metric["PPL"], "Validation_NLL": metric["NLL"],
                "Predicted_Tokens": metric["Predicted_Tokens"], "Original_Validation_PPL": baseline["PPL"],
                "PPL_Delta_vs_Original": metric["PPL"]-baseline["PPL"],
                "PPL_Degradation_%": 100*(metric["PPL"]/baseline["PPL"]-1),
                **reconstruction, "Parameters": params,
                "Merged_Block_Parameters": sm.parameter_count(student.transformer.h[pair]),
                "Parameter_Reduction_%": 100*(1-params/baseline["Parameters"]),
                **latency, "Reference_Latency_ms": reference["Latency_ms"], "Best_Epoch": epoch}
            if not math.isfinite(metric["NLL"]):
                raise ArithmeticError("Invalid validation result")
            if method == "LinearMultiply":
                linear_ppl = metric["PPL"]
            row["PPL_Delta_vs_LinearMultiply"] = metric["PPL"]-linear_ppl
            rows.append(row)
            pd.DataFrame(rows).to_csv(out/"validation_results.csv", index=False)
            save_checkpoint(out/f"block_{method}_{pair}_{pair+1}.pt", teacher, student, pair, method, cfg, epoch)
            if method not in best or metric["NLL"] < best[method]["Validation_NLL"]:
                best[method] = dict(row)
                save_checkpoint(out/f"selected_{method}.pt", teacher, student, pair, method, cfg, epoch)
            print(f"  {method}: PPL={metric['PPL']:.4f}, MSE={reconstruction['Hidden_MSE']:.5g}, params={params:,}", flush=True)
            del student
            cleanup()
    assert parameter_digest(teacher) == teacher_digest, "Original teacher changed"
    table = pd.DataFrame(rows)
    pivot = table.pivot(index="Pair_Start", columns="Method", values="Validation_PPL").reindex(columns=METHODS)
    pivot["Remove_minus_LinearMultiply"] = pivot["Remove"]-pivot["LinearMultiply"]
    pivot["Average_minus_LinearMultiply"] = pivot["Average"]-pivot["LinearMultiply"]
    pivot["NonlinearMultiply_minus_LinearMultiply"] = pivot["NonlinearMultiply"]-pivot["LinearMultiply"]
    pivot.index = [f"{i}-{i+1}" for i in pivot.index]
    pivot.index.name = "Pair"
    pivot.to_csv(out/"validation_pivot.csv")
    selected = pd.DataFrame([best[m] for m in METHODS])
    selected.to_csv(out/"best_pairs.csv", index=False)
    ep._write_json(out/"selection_locked.json", {"methods": [best[m] for m in METHODS],
        "basis": "VALIDATION NLL; training epoch selected only on TRAIN_HOLDOUT",
        "file_sha256": {name: hashlib.sha256((out/name).read_bytes()).hexdigest()
                        for name in ["configuration.json", "best_pairs.csv", *[f"selected_{m}.pt" for m in METHODS]]}})
    print("\n===== BEST VALIDATION PAIR PER METHOD =====\n" + selected.to_string(index=False))
    del teacher
    cleanup()
    test = run_final_test(out) if run_test else None
    return {"output_dir": str(out), "validation": table, "pivot": pivot,
            "best_pairs": selected, "baseline": baseline, "test": test}


def run_final_test(run_dir):
    """Explicit final evaluation of locked checkpoints; never retrains/selects."""
    out = Path(run_dir)
    if (out/"test_started.json").exists() or (out/"test_results.csv").exists():
        raise RuntimeError("TEST already started in this run. Inspect saved results; do not silently repeat TEST.")
    locked = json.loads((out/"selection_locked.json").read_text(encoding="utf-8"))
    for name, expected in locked["file_sha256"].items():
        if hashlib.sha256((out/name).read_bytes()).hexdigest() != expected:
            raise ValueError("Locked selection/config/checkpoint changed: " + name)
    cfg = Config(**json.loads((out/"configuration.json").read_text(encoding="utf-8")))
    env = json.loads((out/"environment.json").read_text(encoding="utf-8"))
    sm.check_transformers_version()
    sm.set_seed(cfg.seed)
    torch.set_num_threads(cfg.cpu_threads)
    import datasets
    dataset = datasets.load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1", revision=cfg.dataset_revision)
    if {n: p._fingerprint for n,p in dataset.items()} != env["dataset_fingerprints"]:
        raise ValueError("Dataset changed since selection")
    tokenizer = sm.AutoTokenizer.from_pretrained(cfg.model_id, revision=env["model_resolved_revision"])
    tokenizer.model_max_length = 10**12
    text = "\n\n".join(t for t in dataset["test"]["text"] if t and t.strip())
    ids = tokenizer(text, return_tensors="pt", add_special_tokens=False)["input_ids"][0]
    if cfg.eval_tokens is not None:
        ids = ids[:cfg.eval_tokens]
    device = "cuda" if torch.cuda.is_available() else "cpu"
    teacher = sm.AutoModelForCausalLM.from_pretrained(cfg.model_id, revision=env["model_resolved_revision"],
        attn_implementation="eager", dtype=torch.float32).to(device).eval().requires_grad_(False)
    teacher.config.use_cache = teacher.generation_config.use_cache = False
    if parameter_digest(teacher) != env["teacher_parameter_sha256"]:
        raise ValueError("Original weights differ from validation run")
    ep._write_json(out/"test_started.json", {"started_utc": datetime.now(timezone.utc).isoformat(),
        "device": device, "torch": str(torch.__version__), "transformers": sm.transformers.__version__})
    torch.save(ids, out/"test_tokens.pt")
    baseline = evaluate_ppl(teacher, ids, cfg)
    rows = [{"Method": "Original", "Pair": None, **baseline, "PPL_Delta_vs_Original": 0.0}]
    ep._write_json(out/"test_progress.json", rows)
    for choice in locked["methods"]:
        student = load_checkpoint(out/f"selected_{choice['Method']}.pt", teacher=teacher)
        metric = evaluate_ppl(student, ids, cfg)
        rows.append({"Method": choice["Method"], "Pair": choice["Pair"], **metric,
                     "PPL_Delta_vs_Original": metric["PPL"]-baseline["PPL"],
                     "Parameters": choice["Parameters"], "Validation_PPL": choice["Validation_PPL"]})
        ep._write_json(out/"test_progress.json", rows)
        del student
        cleanup()
    frame = pd.DataFrame(rows)
    frame.to_csv(out/"test_results.csv", index=False)
    del teacher
    cleanup()
    return frame


In [ ]:
%%writefile /kaggle/working/verify_nonlinear.py
"""Offline checks for the four-way layer-merging notebook.
Random tiny GPT-2 only; no pretrained-quality claims.
"""
import contextlib
import copy
import io
import json
import tempfile
from pathlib import Path
from unittest.mock import patch

import nonlinear_composition as nc
import numpy as np
import torch
from torch.nn import functional as F
from transformers import GPT2Config, GPT2LMHeadModel
from datasets import Dataset, DatasetDict


def run_checks():
    nc.sm.check_transformers_version()
    nc.sm.set_seed(123)
    torch.set_num_threads(2)
    checks = []

    def passed(name):
        checks.append({"check": name, "status": "PASS"})

    cfg = nc.make_config(
        model_id="synthetic-test-only",
        context_length=8,
        eval_stride=4,
        local_tokens=32,
        kd_tokens=32,
        holdout_tokens=16,
        epochs=1,
        patience=1,
        eval_tokens=25,
        hidden_eval_tokens=16,
        latency_seq_len=8,
        latency_repeats=1,
        kl_chunk_tokens=3,
        grad_accumulation=3,
    )
    nc.validate_config(cfg)

    mc = GPT2Config(
        vocab_size=31,
        n_positions=32,
        n_ctx=32,
        n_embd=16,
        n_layer=12,
        n_head=2,
        resid_pdrop=0,
        embd_pdrop=0,
        attn_pdrop=0,
        use_cache=False,
    )
    mc._attn_implementation = "eager"
    teacher = GPT2LMHeadModel(mc).eval().requires_grad_(False)
    original_digest = nc.parameter_digest(teacher)
    ids = torch.randint(0, 31, (1, 8))

    # Teacher targets are the exact two-block compositions for all 11 adjacent pairs.
    for pair in range(11):
        with torch.no_grad(), nc.capture_pair(teacher, pair) as states:
            teacher(ids, use_cache=False)
        h, target = states["h"], states["target"]
        with torch.no_grad():
            first = teacher.transformer.h[pair](h, use_cache=False)[0]
            composed = teacher.transformer.h[pair + 1](first, use_cache=False)[0]
        torch.testing.assert_close(target, composed, rtol=1e-5, atol=1e-7)

        nonlinear = nc.build_student(teacher, pair, "NonlinearMultiply", cfg)
        remove = nc.build_student(teacher, pair, "Remove", cfg)
        torch.testing.assert_close(nonlinear.transformer.h[pair](h)[0], first, rtol=0, atol=0)
        torch.testing.assert_close(remove.transformer.h[pair](h)[0], first, rtol=0, atol=0)
        assert h.shape == target.shape == first.shape

    passed("all_11_teacher_targets_and_remove_nonlinear_initialization")

    # Average really is a parameter-wise 50/50 average of the two source blocks.
    avg = nc.build_student(teacher, 3, "Average", cfg)
    avg_state = avg.transformer.h[3].state_dict()
    a_state = teacher.transformer.h[3].state_dict()
    b_state = teacher.transformer.h[4].state_dict()
    for key, got in avg_state.items():
        if torch.is_floating_point(got) or torch.is_complex(got):
            torch.testing.assert_close(got, (a_state[key] + b_state[key]) * 0.5, rtol=0, atol=0)
        else:
            torch.testing.assert_close(got, a_state[key], rtol=0, atol=0)
    passed("average_is_exact_50_50_parameter_average")

    # All four replacements remain causal.
    for method in nc.METHODS:
        student = nc.build_student(teacher, 3, method, cfg)
        future_changed = ids.clone()
        future_changed[:, 4:] = (future_changed[:, 4:] + 1) % 31
        with torch.no_grad():
            a = student(ids, use_cache=False).logits
            b = student(future_changed, use_cache=False).logits
        torch.testing.assert_close(a[:, :4], b[:, :4], rtol=0, atol=1e-7)
    passed("causal_future_token_perturbation_for_all_four_methods")

    # KL implementation and gradient direction.
    t = torch.randn(2, 7, 31)
    s = torch.randn_like(t).requires_grad_()
    got = nc.distillation_kl(s, t, cfg)
    logp = F.log_softmax(t[:, :-1] / cfg.temperature, -1)
    logq = F.log_softmax(s[:, :-1] / cfg.temperature, -1)
    expected = cfg.temperature**2 * F.kl_div(
        logq, logp, log_target=True, reduction="none"
    ).sum(-1).mean()
    torch.testing.assert_close(got, expected)
    grad1 = torch.autograd.grad(got, s, retain_graph=True)[0]
    grad2 = torch.autograd.grad(expected, s)[0]
    torch.testing.assert_close(grad1, grad2, atol=2e-7, rtol=1e-5)
    assert torch.count_nonzero(grad1[:, -1]) == 0
    passed("KL_direction_temperature_token_mean_chunk_gradient")

    # Only NonlinearMultiply is trainable; gradients flow through the frozen suffix.
    student = nc.build_student(teacher, 4, "NonlinearMultiply", cfg)
    prefix = "transformer.h.4."
    frozen = nc.parameter_digest(student, prefix)
    params = [p for p in student.parameters() if p.requires_grad]
    assert params
    before = [p.detach().clone() for p in params]
    loss = nc.kd_objective(teacher, student, ids, cfg)
    loss.backward()
    assert any(p.grad is not None and torch.count_nonzero(p.grad) for p in params)
    assert all(p.grad is None for p in teacher.parameters())
    assert all(p.grad is None for p in student.parameters() if not p.requires_grad)
    optimizer = torch.optim.AdamW(params, lr=1e-3)
    optimizer.step()
    assert any(not torch.equal(p, b) for p, b in zip(params, before))
    assert nc.parameter_digest(student, prefix) == frozen
    for method in ("LinearMultiply", "Remove", "Average"):
        candidate = nc.build_student(teacher, 4, method, cfg)
        assert not any(p.requires_grad for p in candidate.parameters())
    passed("only_nonlinear_multiply_is_trainable")

    # TRAIN regions remain disjoint.
    parts, regions = nc.split_train(torch.arange(800), cfg)
    token_sets = {key: set(torch.cat(windows).tolist()) for key, windows in parts.items()}
    assert not token_sets["local"] & token_sets["kd"]
    assert not token_sets["local"] & token_sets["holdout"]
    assert not token_sets["kd"] & token_sets["holdout"]
    previous = nc.sm.make_state_windows(torch.arange(800), 32, 8, cfg.seed)
    assert all(torch.equal(a, b) for a, b in zip(previous, parts["local"]))
    passed("disjoint_TRAIN_regions_and_preserved_linear_fit_windows")

    # Parameter budgets. Remove/Average/NonlinearMultiply all keep one GPT2Block;
    # LinearMultiply replaces two blocks with one dxd affine map.
    with torch.device("meta"):
        full = GPT2LMHeadModel(GPT2Config())
    original_params = nc.sm.parameter_count(full)
    one_block = nc.sm.parameter_count(full.transformer.h[0])
    assert original_params == 124439808
    assert original_params - 2 * one_block + 768 * 768 + 768 == 110854656
    assert original_params - one_block == 117351936
    passed("GPT2_parameter_budgets_for_linear_and_one_block_methods")

    # End-to-end orchestrator with mocked external loading.
    data = DatasetDict({
        name: Dataset.from_dict({"text": [name + " " + ("abcdef " * 160)]})
        for name in ("train", "validation", "test")
    })
    events = []

    class Tokenizer:
        model_max_length = 32

        def __call__(self, text, **kwargs):
            events.append("tokenize_" + text.split()[0])
            return {"input_ids": torch.tensor([[ord(c) % 31 for c in text]])}

    with tempfile.TemporaryDirectory(prefix="merging4_checks_") as temp:
        run_cfg = nc.replace(cfg, output_dir=temp)
        with patch("datasets.load_dataset", return_value=data), \
             patch.object(nc.sm.AutoTokenizer, "from_pretrained", return_value=Tokenizer()), \
             patch.object(
                 nc.sm.AutoModelForCausalLM,
                 "from_pretrained",
                 side_effect=lambda *a, **kw: copy.deepcopy(teacher),
             ), \
             contextlib.redirect_stdout(io.StringIO()):
            result = nc.run_experiment(run_cfg)

        out = Path(result["output_dir"])
        assert "tokenize_test" not in events
        assert not (out / "test_results.csv").exists()
        assert len(result["validation"]) == 44
        assert set(result["validation"]["Method"]) == set(nc.METHODS)
        assert result["validation"]["Predicted_Tokens"].eq(24).all()

        history = nc.pd.read_csv(out / "training_history.csv")
        assert len(history) == 22  # 11 NonlinearMultiply fits, epoch 0 + epoch 1
        assert set(history.Method) == {"NonlinearMultiply"}

        for method in nc.METHODS:
            subset = result["validation"].query("Method == @method")
            selected = result["best_pairs"].query("Method == @method").iloc[0]
            assert selected["Validation_NLL"] == subset["Validation_NLL"].min()
            saved = torch.load(out / f"selected_{method}.pt", weights_only=True)
            model = nc.load_checkpoint(out / f"selected_{method}.pt", teacher=teacher)
            full_model = nc.load_checkpoint(
                out / f"block_{method}_{saved['pair']}_{saved['pair'] + 1}.pt",
                teacher=teacher,
            )
            with torch.no_grad():
                torch.testing.assert_close(
                    model(ids).logits, full_model(ids).logits, rtol=0, atol=0
                )
            if method == "NonlinearMultiply":
                h = history[
                    (history.Method == method) & (history.Pair == selected["Pair"])
                ]
                assert saved["best_epoch"] == int(
                    h.loc[h.Holdout_Total_Loss.idxmin(), "Epoch"]
                )
            else:
                assert saved["best_epoch"] is None

        assert nc.parameter_digest(teacher) == original_digest
        passed("all_44_candidates_validation_selection_and_checkpoint_roundtrip")
        passed("RUN_TEST_false_never_tokenizes_or_evaluates_TEST")

        with patch("datasets.load_dataset", return_value=data), \
             patch.object(nc.sm.AutoTokenizer, "from_pretrained", return_value=Tokenizer()), \
             patch.object(
                 nc.sm.AutoModelForCausalLM,
                 "from_pretrained",
                 side_effect=lambda *a, **kw: copy.deepcopy(teacher),
             ):
            final = nc.run_final_test(out)

        assert len(final) == 5 and events.count("tokenize_test") == 1
        assert final.Predicted_Tokens.eq(24).all()
        for method in nc.METHODS:
            assert (
                final[final.Method == method].iloc[0].Pair
                == result["best_pairs"].query("Method == @method").iloc[0].Pair
            )

        try:
            nc.run_final_test(out)
        except RuntimeError as exc:
            assert "already started" in str(exc)
        else:
            raise AssertionError("Repeated TEST must be rejected")

        passed("final_TEST_uses_only_locked_models_and_blocks_accidental_repetition")

    return {
        "status": "PASS",
        "checks": checks,
        "scope": "Offline CPU, random tiny 12-layer GPT-2; not pretrained GPT-2 PPL or Kaggle GPU benchmark",
        "torch": str(torch.__version__),
        "transformers": nc.sm.transformers.__version__,
    }


if __name__ == "__main__":
    report = run_checks()
    Path(__file__).with_name("nonlinear_verification_results.json").write_text(
        json.dumps(report, indent=2), encoding="utf-8"
    )
    print(json.dumps(report, indent=2))


## 3. Self-Checks

Using a randomly initialized small 12-layer GPT-2 that requires no download, we verify the following:

- For all 11 adjacent pairs, the teacher target corresponds to the composition of two layers.
- `Remove` keeps $L_i$ and removes $L_{i+1}$.
- `Average` is an exact 50/50 average of corresponding parameters.
- All four methods preserve causality.
- Only `NonlinearMultiply` is trained.
- TRAIN-region separation, 44 candidates, validation-based pair selection, saving/reloading, and selection without using TEST.

A PASS here only confirms that the software behaves as intended; it does not guarantee the quality of the pretrained GPT-2 experiment.


In [ ]:
import sys, importlib, json
from pathlib import Path

KAGGLE_WORKING = Path('/kaggle/working')
KAGGLE_WORKING.mkdir(parents=True, exist_ok=True)
if str(KAGGLE_WORKING) not in sys.path:
    sys.path.insert(0, str(KAGGLE_WORKING))

import affine_core, evaluation_protocol, nonlinear_composition, verify_nonlinear
for module in (affine_core, evaluation_protocol, nonlinear_composition, verify_nonlinear):
    importlib.reload(module)

report = verify_nonlinear.run_checks()
self_check_path = KAGGLE_WORKING / 'nonlinear_self_checks.json'
self_check_path.write_text(json.dumps(report, indent=2), encoding='utf-8')
print(json.dumps(report, indent=2))
print('Self-check report:', self_check_path)


## 4. Fixed Experimental Conditions

- `quick`: LOCAL 4,096, KD 4,096, HOLDOUT 2,048 tokens, with validation/test capped at 10,000 tokens.
- `full`: LOCAL 20,480, KD 20,480, HOLDOUT 4,096 tokens, using the full validation/test sets.
- **Only NonlinearMultiply is trained for 15 epochs.** With the default `patience=15`, epochs 1–15 are all run, and the checkpoint with the minimum HOLDOUT loss is selected from epochs 0–15, including the untrained epoch 0.
- `LinearMultiply`, `Remove`, and `Average` do not use gradient-based training.
- LOCAL / KD / HOLDOUT are **non-overlapping 512-token windows** from TRAIN. LOCAL is used for affine regression in LinearMultiply; for NonlinearMultiply, LOCAL is used for the hidden-state loss and KD for logit distillation.
- The NonlinearMultiply loss is
  $L=\lambda_h\,\mathrm{mean}[(H_S-H_T)^2]/(\mathrm{mean}[H_T^2]+10^{-8})+\lambda_{KL}\,\tau^2 KL(p_T\|p_S)$.
- $\tau=2$, both hidden and KL loss weights are 1, and the learning rate is `3e-5`.
- The teacher runs under `no_grad()`. In NonlinearMultiply, only the replacement block is trained; later layers remain frozen while gradients pass through them. Dropout is disabled.
- `Average` simply takes a 50/50 average in weight space. No additional weight matching or permutation alignment is performed.
- `Remove` removes $L_{i+1}$ for pair $(i,i+1)$ and keeps $L_i$.
- Evaluation uses float32, eager attention, 512-token windows with stride 256, and no KV cache. TEST is not run by default.

If you change the loss weights or learning rate, fix those choices before looking at validation or test results.


In [ ]:
import json
import torch
import nonlinear_composition as nc

MODE = 'full'
RUN_TEST = False
CFG = nc.make_config(MODE)
# NonlinearMultiply uses 15 epochs by default.
# Example: CFG = nc.make_config(MODE, epochs=15, patience=15, grad_accumulation=2)

nc.validate_config(CFG)
print('GPU available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
print(json.dumps(nc.asdict(CFG), indent=2))

if not torch.cuda.is_available():
    raise RuntimeError(
        'Please change Accelerator to GPU in the Kaggle Notebook Settings.'
    )


## 5. Training and Validation Comparison

There are **44 candidates** in total: 11 pairs × 4 methods.

- LinearMultiply / Remove / Average: evaluated directly without training
- NonlinearMultiply: each pair is trained for 15 epochs, and the best epoch is fixed using HOLDOUT
- The best pair for each method is fixed using validation NLL

TEST is not used at this stage.


In [ ]:
results = nc.run_experiment(CFG, run_test=RUN_TEST)


## 6. Results

If `PPL_Delta_vs_LinearMultiply` is negative, the method has a better PPL than LinearMultiply.
Hidden/KL diagnostics are computed on the first 4,096 validation tokens (8,192 in `full` mode), while PPL is computed over the configured validation set.


In [ ]:
columns = ['Pair', 'Method', 'Validation_PPL', 'PPL_Delta_vs_Original',
           'PPL_Delta_vs_LinearMultiply', 'Hidden_MSE', 'Normalized_Hidden_MSE',
           'Teacher_KL', 'Parameters', 'Parameter_Reduction_%', 'Latency_ms']
display(results['validation'][columns])
display(results['pivot'])
print('===== BEST VALIDATION PAIR PER METHOD =====')
display(results['best_pairs'][['Method', 'Pair', 'Validation_PPL', 'Original_Validation_PPL',
    'PPL_Degradation_%', 'Parameters', 'Parameter_Reduction_%', 'Latency_ms', 'Best_Epoch']])

history_path = Path(results['output_dir']) / 'training_history.csv'
if history_path.exists():
    display(nc.pd.read_csv(history_path))
print('Saved:', results['output_dir'])


## 7. How to Interpret the Results

The four methods serve the following roles:

- **LinearMultiply**: a linear baseline that approximates each of the two layers as a local affine map and composes the maps by matrix multiplication.
- **Remove**: the simplest compression baseline. It discards $L_{i+1}$ and keeps only $L_i$.
- **Average**: averages the same-named parameters of the two Transformer blocks 50/50 to form a single block. No training is used.
- **NonlinearMultiply**: trains a single Transformer block as a nonlinear student for 15 epochs to approximate the composition of the original two layers.

One important caveat is that only NonlinearMultiply uses additional training and knowledge distillation, so the difference among the four methods cannot be attributed solely to nonlinearity. However, they can still be compared as practical layer-merging strategies in terms of PPL, parameter reduction, and latency.

Remove / Average / NonlinearMultiply each retain one GPT2Block, so their parameter counts are similar for GPT-2 small. LinearMultiply replaces the pair with one $768\times768$ affine layer, resulting in a larger parameter reduction.

For NonlinearMultiply, check `training_history.csv`. If HOLDOUT loss continues to decrease through epoch 15, 15 epochs may still be insufficient. If an earlier epoch is selected as best, running through epoch 15 helps avoid the issue of stopping too early, such as after only 3 epochs.


## 8. Final TEST (Run Only When Needed)

After fixing the four methods and all training conditions, explicitly enable the following cell.
Only the **best pair for each method already selected on validation** is loaded from the saved checkpoint and evaluated on TEST, with no retraining or pair reselection.

Do not inspect all 11 pairs on TEST and reselect the best pair. Pair selection must use validation only.

Re-evaluation is prevented for run folders in which TEST has already started. If execution is interrupted, partial progress is saved in `test_progress.json`.


In [ ]:
FINAL_TEST_ON_LOCKED_RUN = True
if FINAL_TEST_ON_LOCKED_RUN:
    display(nc.run_final_test(results['output_dir']))
else:
    print('TEST has not been run.')


In [ ]:
# ============================================================
# External / Out-of-Domain PPL Evaluation
# ============================================================
# IMPORTANT:
# - These data are never used to train the merged layer
# - They are also not used to select the layer pair or epoch
# - Only the checkpoint already selected on validation is evaluated
# ============================================================

import gc
import json
from pathlib import Path

import pandas as pd
import torch
from datasets import load_dataset

import nonlinear_composition as nc


# ------------------------------------------------------------
# Settings
# ------------------------------------------------------------

RUN_DIR = Path(results["output_dir"])

# About 10,000 tokens are enough for an initial check.
# For the final experiment, this can be increased to 50,000 or more.
EXTERNAL_EVAL_TOKENS = 10_000

# Compare all four methods selected on validation
METHODS_TO_TEST = {
    "LinearMultiply",
    "Remove",
    "Average",
    "NonlinearMultiply",
}


# ------------------------------------------------------------
# External datasets
# ------------------------------------------------------------

EXTERNAL_DATASETS = {
    # Book / narrative domain
    "LAMBADA": {
        "path": "EleutherAI/lambada_openai",
        "name": None,
        "split": "test",
        "text_column": "text",
        "streaming": False,
    },

    # Web text domain
    # C4 has no test split, so use the validation split.
    # Since it is not used for training or selection in this experiment,
    # treat it as an external held-out evaluation set.
    "C4": {
        "path": "allenai/c4",
        "name": "en",
        "split": "validation",
        "text_column": "text",
        "streaming": True,
    },
}


# ------------------------------------------------------------
# Dataset -> GPT-2 token IDs
# ------------------------------------------------------------

def load_external_ids(spec, tokenizer, max_tokens):
    kwargs = {
        "split": spec["split"],
        "streaming": spec["streaming"],
    }

    if spec["name"] is None:
        dataset = load_dataset(
            spec["path"],
            **kwargs,
        )
    else:
        dataset = load_dataset(
            spec["path"],
            spec["name"],
            **kwargs,
        )

    text_column = spec["text_column"]

    pieces = []
    total_tokens = 0

    # As in the existing code, insert \n\n between documents
    separator_ids = tokenizer(
        "\n\n",
        add_special_tokens=False
    )["input_ids"]

    for row in dataset:
        text = row[text_column]

        if text is None or not text.strip():
            continue

        # document separator
        if total_tokens > 0 and len(separator_ids) > 0:
            sep = torch.tensor(separator_ids, dtype=torch.long)
            pieces.append(sep)
            total_tokens += len(sep)

        ids = tokenizer(
            text,
            return_tensors="pt",
            add_special_tokens=False,
        )["input_ids"][0].cpu()

        pieces.append(ids)
        total_tokens += ids.numel()

        if total_tokens >= max_tokens:
            break

    if not pieces:
        raise RuntimeError(
            f"No usable text found in {spec['path']}"
        )

    ids = torch.cat(pieces)[:max_tokens]

    if ids.numel() < 2:
        raise RuntimeError("Not enough evaluation tokens")

    return ids


# ------------------------------------------------------------
# Load configuration / locked selection
# ------------------------------------------------------------

cfg = nc.Config(
    **json.loads(
        (RUN_DIR / "configuration.json").read_text(
            encoding="utf-8"
        )
    )
)

env = json.loads(
    (RUN_DIR / "environment.json").read_text(
        encoding="utf-8"
    )
)

locked = json.loads(
    (RUN_DIR / "selection_locked.json").read_text(
        encoding="utf-8"
    )
)


# ------------------------------------------------------------
# Tokenizer
# ------------------------------------------------------------

tokenizer = nc.sm.AutoTokenizer.from_pretrained(
    cfg.model_id,
    revision=env["model_resolved_revision"],
)

tokenizer.model_max_length = 10**12


# ------------------------------------------------------------
# Original GPT-2
# ------------------------------------------------------------

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

teacher = nc.sm.AutoModelForCausalLM.from_pretrained(
    cfg.model_id,
    revision=env["model_resolved_revision"],
    attn_implementation="eager",
    dtype=torch.float32,
).to(device).eval().requires_grad_(False)

teacher.config.use_cache = False
teacher.generation_config.use_cache = False


# Confirm that this is the same GPT-2 used in the original validation run
if nc.parameter_digest(teacher) != env["teacher_parameter_sha256"]:
    raise ValueError(
        "Original GPT-2 weights differ from the training run."
    )


# ------------------------------------------------------------
# External evaluation
# ------------------------------------------------------------

rows = []

for dataset_name, spec in EXTERNAL_DATASETS.items():

    print()
    print("=" * 70)
    print(f"Dataset: {dataset_name}")
    print("=" * 70)

    test_ids = load_external_ids(
        spec,
        tokenizer,
        EXTERNAL_EVAL_TOKENS,
    )

    print(
        f"Evaluation tokens: {test_ids.numel():,}"
    )

    # -------------------------
    # Original GPT-2
    # -------------------------

    baseline = nc.evaluate_ppl(
        teacher,
        test_ids,
        cfg,
    )

    original_ppl = baseline["PPL"]
    original_nll = baseline["NLL"]

    print(
        f"Original GPT-2: "
        f"PPL={original_ppl:.4f}, "
        f"NLL={original_nll:.4f}"
    )

    rows.append({
        "Dataset": dataset_name,
        "Method": "Original",
        "Pair": None,

        "PPL": original_ppl,
        "NLL": original_nll,

        "PPL_Delta_vs_Original": 0.0,
        "PPL_Degradation_%": 0.0,
        "NLL_Delta_vs_Original": 0.0,

        "Predicted_Tokens":
            baseline["Predicted_Tokens"],
    })


    # -------------------------
    # Selected merged models
    # -------------------------

    for choice in locked["methods"]:

        method = choice["Method"]

        if method not in METHODS_TO_TEST:
            continue

        # IMPORTANT:
        # Load only the checkpoint already selected on validation
        # Do not reselect the pair after looking at the external test set
        checkpoint_path = (
            RUN_DIR / f"selected_{method}.pt"
        )

        student = nc.load_checkpoint(
            checkpoint_path,
            teacher=teacher,
        )

        metric = nc.evaluate_ppl(
            student,
            test_ids,
            cfg,
        )

        ppl = metric["PPL"]
        nll = metric["NLL"]

        degradation = (
            100.0
            * (ppl / original_ppl - 1.0)
        )

        row = {
            "Dataset": dataset_name,
            "Method": method,
            "Pair": choice["Pair"],

            "PPL": ppl,
            "NLL": nll,

            "PPL_Delta_vs_Original":
                ppl - original_ppl,

            "PPL_Degradation_%":
                degradation,

            "NLL_Delta_vs_Original":
                nll - original_nll,

            "Predicted_Tokens":
                metric["Predicted_Tokens"],
        }

        rows.append(row)

        print(
            f"{method:20s} "
            f"pair={choice['Pair']:>5s} | "
            f"PPL={ppl:8.4f} | "
            f"degradation={degradation:+7.2f}%"
        )

        del student

        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()


# ------------------------------------------------------------
# Results
# ------------------------------------------------------------

external_results = pd.DataFrame(rows)

output_path = (
    RUN_DIR / "external_generalization_results.csv"
)

external_results.to_csv(
    output_path,
    index=False,
)

print()
print("=" * 70)
print("EXTERNAL GENERALIZATION RESULTS")
print("=" * 70)

display(external_results)


# PPL comparison table
ppl_table = external_results.pivot(
    index="Dataset",
    columns="Method",
    values="PPL",
)

display(ppl_table)


# Degradation (%) comparison table
degradation_table = external_results.pivot(
    index="Dataset",
    columns="Method",
    values="PPL_Degradation_%",
)

display(degradation_table)

print("Saved:", output_path)


# ------------------------------------------------------------
# Cleanup
# ------------------------------------------------------------

del teacher

gc.collect()

if torch.cuda.is_available():
    torch.cuda.empty_cache()

## 9. Save, ZIP, and Reload

The ZIP archive includes the experiment code, configuration, CSV files, tokens, all 44 candidates, and the replacement blocks for the selected model from each method.
The original GPT-2 weights are not included; they are reloaded from the recorded revision.

In Kaggle, files under `/kaggle/working` remain available as **Output**. The following cell creates a ZIP archive of the experiment folder and displays a link in the notebook.


In [ ]:
import shutil
from pathlib import Path
from IPython.display import FileLink, display

run_dir = Path(results['output_dir'])
for name in ['affine_core.py', 'evaluation_protocol.py', 'nonlinear_composition.py',
             'verify_nonlinear.py', 'nonlinear_self_checks.json']:
    src = Path('/kaggle/working') / name
    if src.exists():
        shutil.copy2(src, run_dir / name)

archive = shutil.make_archive(str(run_dir), 'zip', root_dir=run_dir)
print('ZIP saved to:', archive)
display(FileLink(archive))


In [ ]:
# Example: load the validation-selected NonlinearMultiply model (definition only)
def load_selected_nonlinear(run_dir, device=None):
    if device is None:
        device = 'cuda' if nc.torch.cuda.is_available() else 'cpu'
    return nc.load_checkpoint(
        Path(run_dir) / 'selected_NonlinearMultiply.pt',
        device=device
    )

# model = load_selected_nonlinear(results['output_dir'])
# output = model(input_ids, use_cache=False)


## References

- Hinton et al., *Distilling the Knowledge in a Neural Network*
- Transformers v4.57.6 GPT-2 implementation
- PPL and affine-regression baseline: original `merging_comparison_colab.ipynb`

In this notebook, `Average` is defined as a **simple 50/50 average of corresponding parameters**, while `Remove` is defined as **removing the later layer $L_{i+1}$**.
